In [1]:
# 1. Imports

from pathlib import Path
import time


# ============================================================
# EXTERNE BIBLIOTHEKEN - Datenverarbeitung & Analyse
# ============================================================
import numpy as np                           # Numerische Berechnungen
import pandas as pd                          # Datenanalyse und TimeSeriesManagement
import ccxt                                  # Kryptobörsen-API (Binance etc.)

from dataclasses import dataclass
import configparser
import hmac
import hashlib
import requests
import pyarrow

In [3]:
# 2. Configuration

# Change run date
RUN_DATE = "2026-03-13"
TOP_N = 100
VS_CURRENCY = "usd"


PROJECT_ROOT = Path.cwd().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FETCHED_DIR = RAW_DIR / "fetched"
PROCESSED_DIR = DATA_DIR / "processed"

ASSETS_PATH = FETCHED_DIR / "assets.parquet"
ASSET_DAILY_PATH = FETCHED_DIR / "asset_daily"  
ASSET_OHLCV_5M_DIR = FETCHED_DIR / "asset_ohlcv_5m"

SRC_PATH = PROJECT_ROOT / "Src"
SECRET_PATH = SRC_PATH / "secret.cfg"

TARGET_VOLUME_EXCHANGE = "binance"
USD_LIKE_QUOTES = ["USDT", "USDC", "USD"]

OHLCV_INTERVAL = "5m"
BINANCE_KLINE_LIMIT = 1000
OHLCV_START_DATE = RUN_DATE
OHLCV_END_DATE_EXCLUSIVE = "2026-03-14"

def controlpath():
    print(f"Projektverzeichnis:            {PROJECT_ROOT}")
    print(f"Datenverzeichnis:              {DATA_DIR}")
    print(f"Rohdatenverzeichnis:           {RAW_DIR}")
    print(f"Gefetchte Daten:               {FETCHED_DIR}")
    print(f"Verarbeitete Daten:            {PROCESSED_DIR}")
    print(f"Source-Code:                   {SRC_PATH}")
    print(f"Secret:                        {SECRET_PATH}")
    print()

    checks = {
        "Assets-Datei":              ASSETS_PATH,
        "Asset-Daily-Verzeichnis":   ASSET_DAILY_PATH,
        "Asset-OHLCV-5M-Verzeichnis": ASSET_OHLCV_5M_DIR,
    }

    for name, path in checks.items():
        exists = path.exists()
        symbol = "✅" if exists else "❌ FEHLT"
        print(f"  {symbol}  {name}: {path}")

        # Falls Ordner → Inhalt anzeigen
        if exists and path.is_dir():
            files = sorted(path.glob("*.parquet"))
            if files:
                for f in files:
                    print(f"       📄 {f.name}")
            else:
                print(f"       ⚠️  Ordner ist leer — keine .parquet Dateien!")


OHLCV_5M_COLUMNS = [
    "timestamp",
    "asset_id",
    "exchange",
    "market_symbol",
    "open",
    "high",
    "low",
    "close",
    "volume_base",
    "volume_quote",
    "trade_count",
]

controlpath()

Projektverzeichnis:            c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit
Datenverzeichnis:              c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data
Rohdatenverzeichnis:           c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw
Gefetchte Daten:               c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched
Verarbeitete Daten:            c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\processed
Source-Code:                   c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\Src
Secret:                        c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\Src\secret.cfg

  ✅  Assets-Datei: c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\assets.parquet
  ✅  Asset-Daily-Verzeichnis: c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\asset_daily
       📄 asset_daily.parquet
  ✅  Asset-

In [ ]:
# =========================
# Konfiguration
# =========================
BASE_URL = "https://data-api.binance.vision"
INTERVAL = "5m"          
SLEEP_BETWEEN_REQUESTS = 0.15
# globaler Zeit-Offset (Serverzeit - lokale Zeit) in ms
TIME_OFFSET_MS = 0
TESTNET_REST = "https://testnet.binance.vision"

@dataclass
class BinanceKeys:
    api_key: str
    secret_key: str

def load_keys() -> BinanceKeys:
    """ Lädt die API-Schlüssel aus der Konfigurationsdatei. """
    cfg = configparser.ConfigParser()
    if not cfg.read(SECRET_PATH):
        raise FileNotFoundError(f"Konnte {SECRET_PATH} nicht lesen.")
    return BinanceKeys(
        api_key=cfg["BINANCE"]["TEST_API_KEY"].strip(),
        secret_key=cfg["BINANCE"]["TEST_SECRET_KEY"].strip()
    )

def _local_ms() -> int:
    return int(time.time() * 1000)

def _ts_ms() -> int:
    # Server synchronisiert: lokale Zeit + Offset
    return _local_ms() + TIME_OFFSET_MS

def _sign(secret_key: str, query_string: str) -> str:
    """ Berechnet die HMAC-SHA256-Signatur für die gegebene Query-String mit dem Secret Key.
    Diese Signatur wird für authentifizierte Anfragen an die Binance-API benötigt."""
    return hmac.new(
        secret_key.encode("utf-8"),
        query_string.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

def public_get(path: str, params: dict | None = None):
    """ Führt eine GET-Anfrage an die Binance-API durch und gibt die JSON-Antwort zurück."""
    url = f"{TESTNET_REST}{path}"
    r = requests.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Public Fehler {r.status_code}: {r.text}")
    return r.json()

def sync_server_time_offset():
    """
    Holt die Binance-Serverzeit und berechnet den Offset.
    Offset = serverTime - localTime
    """
    global TIME_OFFSET_MS
    try:
        server = public_get("/api/v3/time")
        server_ms = int(server["serverTime"])
        local_ms = _local_ms()
        TIME_OFFSET_MS = server_ms - local_ms
        print(f"[TIME] Offset synchronisiert: {TIME_OFFSET_MS} ms (server-local)")
    except Exception as e:
        print(f"[TIME] Konnte Offset nicht synchronisieren: {e}")

def json_to_parquet(data: list[dict]) -> None:
    """
    Konvertiert JSON-Daten zu Parquet.
    - Gleicher Monat → in bestehendes File schreiben
    - Neuer Monat   → neues File erstellen
    
    Erwartet: data hat eine "timestamp" Spalte (UTC)
    """
    ASSET_OHLCV_5M_DIR.mkdir(parents=True, exist_ok=True)
    
    df = pd.DataFrame(data)
    df["timestamp"] = df["timestamp"].dt.tz_convert("UTC")
    
    # Nach Monat gruppieren
    df["_year"]  = df["timestamp"].dt.year
    df["_month"] = df["timestamp"].dt.month

    for (year, month), month_df in df.groupby(["_year", "_month"]):
        month_df = month_df.drop(columns=["_year", "_month"])
        
        out = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        if out.exists():
            # ── Monat existiert bereits → zusammenführen ──
            df_existing = pd.read_parquet(out)
            df_all = pd.concat([df_existing, month_df], ignore_index=True)
            df_all = (
                df_all
                .drop_duplicates(subset=["timestamp", "market_symbol"])
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  Aktualisiert: {out.name}  ({len(df_all)} Zeilen)")
        else:
            # ── Neuer Monat → frisch erstellen ──
            df_all = (
                month_df
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  Erstellt:     {out.name}  ({len(df_all)} Zeilen)")

        df_all.to_parquet(out, index=False)

def parse_kline(raw: dict) -> dict | None:
    """
    Einen einzelnen Binance Kline WebSocket Event parsen.
    Gibt None zurück wenn Kerze noch nicht abgeschlossen ist.
    """
    k = raw["k"]

    # Nur abgeschlossene Kerzen speichern
    if not k["x"]:
        return None

    return {
        "timestamp":     pd.to_datetime(int(k["t"]), unit="ms", utc=True),
        "market_symbol": raw["s"],          # von äusserer Ebene
        "open":          float(k["o"]),
        "high":          float(k["h"]),
        "low":           float(k["l"]),
        "close":         float(k["c"]),
        "volume_base":   float(k["v"]),
        "volume_quote":  float(k["q"]),
        "trade_count":   int(k["n"]),
    }


def parse_klines(raw_list: list[dict]) -> list[dict]:
    """Liste von Binance Kline Events parsen — unfertige Kerzen überspringen."""
    parsed = []
    for raw in raw_list:
        result = parse_kline(raw)
        if result is not None:
            parsed.append(result)
    return parsed


def klines_to_parquet(raw_list: list[dict]) -> None:
    """
    Binance Kline-Daten parsen und als Parquet speichern.
    - Gleicher Monat → in bestehendes File schreiben
    - Neuer Monat   → neues File erstellen
    """
    ASSET_OHLCV_5M_DIR.mkdir(parents=True, exist_ok=True)

    parsed = parse_klines(raw_list)

    if not parsed:
        print("  Keine abgeschlossenen Kerzen.")
        return

    df = pd.DataFrame(parsed)
    df["_year"]  = df["timestamp"].dt.year
    df["_month"] = df["timestamp"].dt.month

    for (year, month), month_df in df.groupby(["_year", "_month"]):
        month_df = month_df.drop(columns=["_year", "_month"])

        out = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        if out.exists():
            df_existing = pd.read_parquet(out)
            df_all = pd.concat([df_existing, month_df], ignore_index=True)
            df_all = (
                df_all
                .drop_duplicates(subset=["timestamp", "market_symbol"])
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  Aktualisiert: {out.name}  ({len(df_all)} Zeilen)")
        else:
            df_all = (
                month_df
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  Erstellt:     {out.name}  ({len(df_all)} Zeilen)")

        df_all.to_parquet(out, index=False)


def fetch_ohlcv_for_day() -> None:
    print("Starte OHLCV-Datenbeschaffung...")
    parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))
    print(f"Gefundene Monatsdateien: {[f.name for f in parquet_files]}")

    for file in parquet_files:
        df_month = pd.read_parquet(file)
        df_month["date"] = pd.to_datetime(df_month["date"]).dt.date

        for current_date, df_day in df_month.groupby("date"):
            tickers = df_day["symbol"].dropna().unique().tolist()
            print(f"{current_date}: {len(tickers)} Ticker")

            all_rows = []

            for symbol in tickers:
                print(f"  Lade {symbol} für {current_date}")
                try:
                    # REST API gibt Liste von Listen zurück → direkt parsen
                    raw = public_get("/api/v3/klines", params={
                        "symbol":    symbol,
                        "interval":  INTERVAL,
                        "startTime": int(pd.Timestamp(current_date, tz="UTC").timestamp() * 1000),
                        "endTime":   int((pd.Timestamp(current_date, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000),
                        "limit":     1000,
                    })

                    # REST Format ist anders als WebSocket Format!
                    for row in raw:
                        all_rows.append({
                            "timestamp":     pd.to_datetime(int(row[0]), unit="ms", utc=True),
                            "market_symbol": symbol,
                            "open":          float(row[1]),
                            "high":          float(row[2]),
                            "low":           float(row[3]),
                            "close":         float(row[4]),
                            "volume_base":   float(row[5]),
                            "volume_quote":  float(row[7]),
                            "trade_count":   int(row[8]),
                        })

                    time.sleep(SLEEP_BETWEEN_REQUESTS)
                    print(f"  {len(raw)} Kerzen geladen.")

                except Exception as e:
                    print(f"  Fehler {symbol}: {e}")

            if all_rows:
                # REST Daten direkt → json_to_parquet (kein "k" Objekt)
                json_to_parquet(all_rows)


fetch_ohlcv_for_day()




,asset_id,symbol,market_symbol
0,1,BTC,BTC/USDT
1,2,ETH,ETH/USDT
2,3,USDT,USDT/USD
3,4,XRP,XRP/USDT
4,5,BNB,BNB/USDT
...,...,...,...
95,96,FET,FET/USDT
96,97,VIRTUAL,VIRTUAL/USDT
97,98,PENGU,PENGU/USDT
98,99,CAKE,CAKE/USDT


Fetched 289 rows for BTC (BTC/USDT)
Fetched 289 rows for ETH (ETH/USDT)
Fetched 289 rows for USDT (USDT/USD)
Fetched 289 rows for XRP (XRP/USDT)
Fetched 289 rows for BNB (BNB/USDT)
Fetched 289 rows for USDC (USDC/USDT)
Fetched 289 rows for SOL (SOL/USDT)
Fetched 289 rows for TRX (TRX/USDT)
OHLCV fetch failed for FIGR_HELOC (nan): can only concatenate str (not "float") to str
Fetched 289 rows for DOGE (DOGE/USDT)
OHLCV fetch failed for WBT (nan): can only concatenate str (not "float") to str
OHLCV fetch failed for USDS (nan): can only concatenate str (not "float") to str
Fetched 289 rows for ADA (ADA/USDT)
OHLCV fetch failed for HYPE (nan): can only concatenate str (not "float") to str
Fetched 289 rows for BCH (BCH/USDT)
OHLCV fetch failed for LEO (nan): can only concatenate str (not "float") to str
Fetched 289 rows for LINK (LINK/USDT)
OHLCV fetch failed for XMR (nan): can only concatenate str (not "float") to str
Fetched 289 rows for USDE (USDE/USDT)
OHLCV fetch failed for CC (nan): c

,timestamp,asset_id,exchange,market_symbol,open,high,low,close,volume_base,volume_quote,trade_count
0,2026-03-13 00:00:00+00:00,1,binance,BTC/USDT,70541.3400,70541.3400,70386.0100,70480.5400,177.17450,1.249181e+07,16395
1,2026-03-13 00:05:00+00:00,1,binance,BTC/USDT,70480.5400,70633.7900,70480.5400,70633.7800,47.60617,3.360137e+06,16210
2,2026-03-13 00:10:00+00:00,1,binance,BTC/USDT,70633.7900,71118.5900,70633.7800,71118.5800,438.21589,3.108579e+07,45915
3,2026-03-13 00:15:00+00:00,1,binance,BTC/USDT,71118.5800,71680.0000,71105.3300,71270.1800,594.63126,4.246651e+07,76327
4,2026-03-13 00:20:00+00:00,1,binance,BTC/USDT,71270.1800,71759.6000,71260.4200,71530.0000,367.08836,2.628977e+07,63933
...,...,...,...,...,...,...,...,...,...,...,...
17335,2026-03-13 23:40:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,1557.00000,1.556066e+03,16
17336,2026-03-13 23:45:00+00:00,100,binance,TUSD/USDT,0.9995,0.9995,0.9994,0.9994,69.00000,6.895980e+01,7
17337,2026-03-13 23:50:00+00:00,100,binance,TUSD/USDT,0.9994,0.9995,0.9994,0.9995,152.00000,1.519197e+02,7
17338,2026-03-13 23:55:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,40.00000,3.997600e+01,5


,asset_id,symbol,market_symbol,error_message
0,9,FIGR_HELOC,NaN,"TypeError('can only concatenate str (not ""floa..."
1,11,WBT,NaN,"TypeError('can only concatenate str (not ""floa..."
2,12,USDS,NaN,"TypeError('can only concatenate str (not ""floa..."
3,14,HYPE,NaN,"TypeError('can only concatenate str (not ""floa..."
4,16,LEO,NaN,"TypeError('can only concatenate str (not ""floa..."
5,18,XMR,NaN,"TypeError('can only concatenate str (not ""floa..."
6,20,CC,NaN,"TypeError('can only concatenate str (not ""floa..."
7,27,RAIN,NaN,"TypeError('can only concatenate str (not ""floa..."
8,28,DAI,NaN,"TypeError('can only concatenate str (not ""floa..."
9,30,PYUSD,NaN,"TypeError('can only concatenate str (not ""floa..."


asset_ohlcv_5m validation checks passed.


Saved monthly OHLCV 5m files:
../../data/raw/fetched/asset_ohlcv_5m/asset_ohlcv_5m_2026-03.parquet


OHLCV 5m summary
OHLCV_START_DATE: 2026-03-13
OHLCV_END_DATE_EXCLUSIVE: 2026-03-14
rows: 17340
assets with errors: 40
OHLCV directory: ../../data/raw/fetched/asset_ohlcv_5m


,timestamp,asset_id,exchange,market_symbol,open,high,low,close,volume_base,volume_quote,trade_count,year,month
0,2026-03-13 00:00:00+00:00,1,binance,BTC/USDT,70541.3400,70541.3400,70386.0100,70480.5400,177.17450,1.249181e+07,16395,2026,3
1,2026-03-13 00:05:00+00:00,1,binance,BTC/USDT,70480.5400,70633.7900,70480.5400,70633.7800,47.60617,3.360137e+06,16210,2026,3
2,2026-03-13 00:10:00+00:00,1,binance,BTC/USDT,70633.7900,71118.5900,70633.7800,71118.5800,438.21589,3.108579e+07,45915,2026,3
3,2026-03-13 00:15:00+00:00,1,binance,BTC/USDT,71118.5800,71680.0000,71105.3300,71270.1800,594.63126,4.246651e+07,76327,2026,3
4,2026-03-13 00:20:00+00:00,1,binance,BTC/USDT,71270.1800,71759.6000,71260.4200,71530.0000,367.08836,2.628977e+07,63933,2026,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17335,2026-03-13 23:40:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,1557.00000,1.556066e+03,16,2026,3
17336,2026-03-13 23:45:00+00:00,100,binance,TUSD/USDT,0.9995,0.9995,0.9994,0.9994,69.00000,6.895980e+01,7,2026,3
17337,2026-03-13 23:50:00+00:00,100,binance,TUSD/USDT,0.9994,0.9995,0.9994,0.9995,152.00000,1.519197e+02,7,2026,3
17338,2026-03-13 23:55:00+00:00,100,binance,TUSD/USDT,0.9994,0.9994,0.9994,0.9994,40.00000,3.997600e+01,5,2026,3


,asset_id,symbol,market_symbol,error_message
0,9,FIGR_HELOC,NaN,"TypeError('can only concatenate str (not ""floa..."
1,11,WBT,NaN,"TypeError('can only concatenate str (not ""floa..."
2,12,USDS,NaN,"TypeError('can only concatenate str (not ""floa..."
3,14,HYPE,NaN,"TypeError('can only concatenate str (not ""floa..."
4,16,LEO,NaN,"TypeError('can only concatenate str (not ""floa..."
5,18,XMR,NaN,"TypeError('can only concatenate str (not ""floa..."
6,20,CC,NaN,"TypeError('can only concatenate str (not ""floa..."
7,27,RAIN,NaN,"TypeError('can only concatenate str (not ""floa..."
8,28,DAI,NaN,"TypeError('can only concatenate str (not ""floa..."
9,30,PYUSD,NaN,"TypeError('can only concatenate str (not ""floa..."
